# Lakebase Console

One notebook for working with the Lakebase Postgres instance from this workspace:
connect, inspect the corpus, search it, and grant the deployed app access to the
secrets it needs.

**Run on any cluster.** Cells 1–4 need nothing beyond what Databricks ships
(`psycopg2`, `pandas`). The optional text-search section at the end installs one
small package.

### Why the connection works this way

The instance page offers only a SQL editor, but Lakebase is an ordinary Postgres
endpoint reached over TLS — a workspace notebook is just another client. The
connection URL lives in the `database/lakebase-url` secret.

`dbutils.secrets.get()` returns the **decoded** value, so it can be handed to
`psycopg2.connect()` directly. (The SDK's `w.secrets.get_secret()` differs — it
returns base64 and needs decoding first.)

In [ ]:
import psycopg2
import pandas as pd
from urllib.parse import urlparse, unquote

LAKEBASE_URL = dbutils.secrets.get(scope="database", key="lakebase-url")

def connect():
    """Open a Lakebase connection. Call again if a cell fails on a dropped
    connection - idle Postgres connections are closed by the server."""
    p = urlparse(LAKEBASE_URL)
    return psycopg2.connect(
        host=p.hostname,
        port=p.port or 5432,
        dbname=p.path.lstrip("/"),
        user=p.username,
        # unquote: psycopg2 keyword args take the literal password, but the URL
        # carries it percent-encoded. Without this a password containing %, @,
        # : or / fails as "password authentication failed".
        password=unquote(p.password),
        sslmode="require",
    )

def q(sql, params=None):
    """Run a query and return a DataFrame."""
    with connect() as conn:
        return pd.read_sql(sql, conn, params=params)

_p = urlparse(LAKEBASE_URL)
print(f"host    : {_p.hostname}")
print(f"database: {_p.path.lstrip('/')}")
print(f"user    : {_p.username}")

info = q("SELECT current_user AS usr, current_database() AS db, version() AS v")
print(f"\nconnected as {info.usr[0]} to {info.db[0]}")
print(info.v[0].split(" on ")[0])

In [ ]:
display(q("""
    SELECT 'watchlist'                    AS table_name, COUNT(*) AS rows FROM watchlist
    UNION ALL SELECT 'ticker_news_documents',       COUNT(*) FROM ticker_news_documents
    UNION ALL SELECT 'ticker_news_embeddings',      COUNT(*) FROM ticker_news_embeddings
    UNION ALL SELECT 'ticker_news_chunk_embeddings',COUNT(*) FROM ticker_news_chunk_embeddings
    ORDER BY table_name
"""))

# Coverage: any document without an embedding means the pipeline has not
# finished. Re-run notebooks/lakebase_embeddings.py to close the gap.
display(q("""
    SELECT COUNT(*) AS documents_without_embeddings
    FROM ticker_news_documents d
    LEFT JOIN ticker_news_embeddings e ON e.id = d.id
    WHERE e.id IS NULL
"""))

In [ ]:
# Confirms the vectors are real pgvector values of the expected width, and that
# the HNSW indexes exist. A seq scan in EXPLAIN is normal at this corpus size -
# the planner only prefers the index once the table is much larger.
display(q("""
    SELECT pg_typeof(embedding)::text AS column_type,
           vector_dims(embedding)     AS dims,
           model_name,
           COUNT(*) OVER ()           AS total_rows
    FROM ticker_news_embeddings LIMIT 1
"""))

display(q("""
    SELECT tablename, indexname
    FROM pg_indexes
    WHERE schemaname = 'public' AND indexname LIKE '%embedding%'
    ORDER BY tablename
"""))

display(q("""
    SELECT ticker, COUNT(*) AS articles
    FROM ticker_news_embeddings GROUP BY ticker ORDER BY ticker
"""))

In [ ]:
# Semantic search WITHOUT loading a model: use a stored vector as the probe.
# Answers "what else is like this article?" - and needs no extra packages,
# because the query vector already exists in the table.
SEED_TITLE_LIKE = "%Microsoft%"   # change this to steer the search

display(q("""
    WITH probe AS (
        SELECT embedding, title
        FROM ticker_news_embeddings
        WHERE title ILIKE %(pat)s
        LIMIT 1
    )
    SELECT e.ticker,
           LEFT(e.title, 70) AS title,
           ROUND((1 - (e.embedding <=> probe.embedding))::numeric, 4) AS similarity
    FROM ticker_news_embeddings e, probe
    ORDER BY e.embedding <=> probe.embedding
    LIMIT 10
""", {"pat": SEED_TITLE_LIKE}))

## 5. Grant the deployed app access to the secrets

A Databricks App runs as **its own service principal**, not as you. `app.yaml`
passes only the secret scope and key *names*; the principal resolves the values
at runtime. Without READ on both scopes every request fails with a permissions
error.

Skip this until the app exists — the service principal is created with it.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import workspace
from databricks.sdk.errors import DatabricksError

w = WorkspaceClient()

apps = list(w.apps.list())
if not apps:
    print("No apps found. Deploy the app first (Compute > Apps), then re-run.")
else:
    for a in apps:
        print(f"{a.name}  sp_client_id={a.service_principal_client_id}  "
              f"sp_name={a.service_principal_name}")

    # Set explicitly if you have more than one app.
    APP_NAME = apps[0].name
    app = w.apps.get(name=APP_NAME)
    principal = app.service_principal_client_id
    print(f"\nGranting READ to {APP_NAME} (principal {principal})")

    for scope in ("database", "massive"):
        try:
            w.secrets.put_acl(scope=scope, principal=principal,
                              permission=workspace.AclPermission.READ)
            print(f"  granted READ on {scope}")
        except DatabricksError as e:
            # If this rejects the principal as unknown, retry with
            # app.service_principal_name instead of the client id.
            print(f"  {scope}: {e}")

    for scope in ("database", "massive"):
        print(f"\n{scope}:")
        for acl in w.secrets.list_acls(scope=scope):
            mark = "  <-- the app" if acl.principal == principal else ""
            print(f"  {acl.principal}: {acl.permission}{mark}")

## 6. Optional — search by free text

Everything above searches using vectors already in the table. To search by an
arbitrary phrase, the query has to be embedded with **the same model** that
produced the stored vectors (`all-MiniLM-L6-v2`, 384 dims) — a different model
puts the query in a different space and the distances become meaningless.

`fastembed` runs that model's ONNX export on CPU, instead of
`sentence-transformers` and its ~2.5GB of torch. Measured against vectors this
project's pipeline wrote, the two agree to cosine 0.99+, so rankings match.

`restartPython()` below **clears every variable defined above**, so the search
cell reconnects on its own.

In [ ]:
%pip install --quiet fastembed

In [ ]:
dbutils.library.restartPython()

In [ ]:
# Self-contained: restartPython() above cleared the earlier cells' state.
import os
import psycopg2
import pandas as pd
from urllib.parse import urlparse, unquote
from fastembed import TextEmbedding

os.environ.setdefault("FASTEMBED_CACHE_PATH", "/tmp/.cache/fastembed")

QUERY = "AI datacenter capital spending"   # <- edit this
MODE  = "chunks"                           # "chunks" (passages) or "articles"
LIMIT = 5

_p = urlparse(dbutils.secrets.get(scope="database", key="lakebase-url"))
conn = psycopg2.connect(host=_p.hostname, port=_p.port or 5432,
                        dbname=_p.path.lstrip("/"), user=_p.username,
                        password=unquote(_p.password), sslmode="require")

# Downloads ~50MB on first use, then cached.
model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
vec = "[" + ",".join(f"{float(x):.7g}" for x in list(model.embed([QUERY]))[0]) + "]"

if MODE == "chunks":
    sql = """
        SELECT c.ticker, LEFT(d.title, 55) AS title, c.chunk_index,
               LEFT(c.chunk_text, 160) AS passage,
               ROUND((1 - (c.embedding <=> %(v)s::vector))::numeric, 4) AS similarity
        FROM ticker_news_chunk_embeddings c
        JOIN ticker_news_documents d ON d.id = c.article_id
        ORDER BY c.embedding <=> %(v)s::vector
        LIMIT %(n)s
    """
else:
    sql = """
        SELECT e.ticker, LEFT(e.title, 70) AS title, d.article_url,
               ROUND((1 - (e.embedding <=> %(v)s::vector))::numeric, 4) AS similarity
        FROM ticker_news_embeddings e
        JOIN ticker_news_documents d ON d.id = e.id
        ORDER BY e.embedding <=> %(v)s::vector
        LIMIT %(n)s
    """

print(f"query: {QUERY!r}  mode={MODE}")
display(pd.read_sql(sql, conn, params={"v": vec, "n": LIMIT}))
conn.close()